# RAG 聊天机器人 - 模板版本

## 功能介绍

本笔记本使用 **HuggingFace Inference API** 创建一个能够回答文档相关问题的聊天机器人。

**这是一个空白模板。** 请将下方的占位符替换为您自己的主题、文档和角色设定。

> 在此描述您的工作坊场景。例如：您正在主持一个关于[您的主题]的工作坊，学生正在学习[您的学习目标]。

**您需要：**
- 一个免费的 HuggingFace API 令牌（约300次请求/小时）
- 网络连接

**费用：** 100% 免费

**支持的文档格式：**
- Google 云端硬盘中的 **PDF** 文件
- Google 云端硬盘中的 **DOCX** 文件（Word 文档）
- **Google 文档**（自动导出）

**功能特点：**
- 来源引用（查看使用了哪些文档）
- 对话记忆（支持追问）
- 30秒超时保护

---

**步骤：** 共9步 | **时间：** 约5分钟完成设置

---
## 第1步：安装依赖库

安装所需工具，大约需要30秒。

In [ ]:
# 安装所有必需的软件包
!pip install -q chromadb gradio pypdf sentence-transformers huggingface_hub gdown python-docx

print("✅ 所有依赖库安装成功！")
print("\n📄 支持的文档类型：")
print("   • PDF 文件")
print("   • DOCX 文件（Word 文档）")
print("   • Google 文档（通过公开链接）")
print("\nℹ️  注意：您可能会看到依赖警告 - 这些不影响使用。")

---
## 第2步：加载依赖库

加载刚刚安装的工具。

In [ ]:
import os
import time
import asyncio
import requests
import gradio as gr
import gdown
from huggingface_hub import InferenceClient
from pypdf import PdfReader
from docx import Document as DocxDocument
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import warnings
warnings.filterwarnings('ignore')

print("✅ 依赖库导入成功！")
print("📄 已准备好处理：PDF、DOCX、Google 文档")

---
## 第3步：下载文档

下载聊天机器人所需的文档。

**支持的格式：**
- Google 云端硬盘中的 **PDF** 文件
- Google 云端硬盘中的 **DOCX** 文件
- **Google 文档**（自动导出为文本）

**需要设置** - 运行前请在下方添加您的文档链接。

In [ ]:
# ============================================================================
# 文档来源 - 在此添加您的文档 ✏️
# ============================================================================
# 
# 支持的格式：
#   1. PDF 文件：    ("文件名.pdf", "GOOGLE_DRIVE_文件ID", "pdf")
#   2. DOCX 文件：   ("文件名.docx", "GOOGLE_DRIVE_文件ID", "docx")
#   3. Google 文档：  ("文件名.txt", "GOOGLE_DOC_ID", "gdoc")
#
# 如何获取 ID：
#   PDF/DOCX: https://drive.google.com/file/d/文件ID在这里/view
#   Google 文档: https://docs.google.com/document/d/文档ID在这里/edit
#
# ============================================================================

DOCUMENTS = [
    # Google 云端硬盘中的 PDF 文件
    # ("Your_Document.pdf", "YOUR_GOOGLE_DRIVE_FILE_ID", "pdf"),

    # Google 云端硬盘中的 DOCX 文件
    # ("Your_Document.docx", "YOUR_GOOGLE_DRIVE_FILE_ID", "docx"),

    # Google 文档（导出为文本）
    # ("Your_Document.txt", "YOUR_GOOGLE_DOC_ID", "gdoc"),

    # --- 示例文档 ---
    ("Architecture of resonance.txt", "1ngHWgVrz-GsXa3j68FWgqzKv3MWj7Zf3drsl93ye-GI", "gdoc"),
    ("Destroying the world.txt", "1aCwtt8Rqr6M5J2IkToY7nn2xy-vzI3vcSm8_mX-Yg-s", "gdoc"),
    ("Cinematic storytelling.txt", "1KrS2O5JFHllo0lfPgnmt2WLUcnklXiut67_HMdN1w3I", "gdoc"),
    ("Harmony in Motion.txt", "1UqYPuubwjEvXUDsw0dWunSOQ9Ulnra4DAOYKNqZfXsc", "gdoc"),
]

# ============================================================================
# 下载文档
# ============================================================================

DOCS_DIR = "/content/docs"
os.makedirs(DOCS_DIR, exist_ok=True)

print("📥 正在下载文档...\n")
DOCUMENT_PATHS = []

for filename, file_id, doc_type in DOCUMENTS:
    output_path = os.path.join(DOCS_DIR, filename)
    print(f"正在下载：{filename}（{doc_type.upper()}）")
    
    try:
        if doc_type == "pdf" or doc_type == "docx":
            # 从 Google 云端硬盘下载
            url = f"https://drive.google.com/uc?id={file_id}"
            gdown.download(url, output_path, quiet=True)
            
        elif doc_type == "gdoc":
            # 将 Google 文档导出为纯文本
            url = f"https://docs.google.com/document/d/{file_id}/export?format=txt"
            response = requests.get(url)
            if response.status_code == 200:
                with open(output_path, 'w', encoding='utf-8') as f:
                    f.write(response.text)
            else:
                print(f"  ❌ 导出 Google 文档失败（状态码 {response.status_code}）")
                continue
        
        if os.path.exists(output_path):
            DOCUMENT_PATHS.append((output_path, doc_type))
            print(f"  ✅ 下载成功")
        else:
            print(f"  ❌ 下载失败")
            
    except Exception as e:
        print(f"  ❌ 错误：{str(e)}")

print(f"\n✅ 已下载 {len(DOCUMENT_PATHS)} 个文档")
print(f"📁 存储位置：{DOCS_DIR}")

if len(DOCUMENT_PATHS) == 0:
    print("\n⚠️  警告：没有下载到任何文档！")
    print("请检查：文件是否已设置为"知道链接的任何人都可查看"？")
    print("💡 提示：请在上方的 DOCUMENTS 列表中添加您的文档。")

---
## 第4步：配置 - 只需填写 API 令牌！✏️

**⚠️ 您只需在下方添加 HuggingFace 令牌**

### 获取免费 HuggingFace 令牌：
1. 访问：https://huggingface.co/
2. 注册免费账号（邮箱 + 密码）
3. 访问：https://huggingface.co/settings/tokens
4. 点击 "Create new token"（创建新令牌）
5. 命名为："RAG Chatbot"
6. 类型：选择 **"Fine-grained"**，启用 **"Make calls to Inference Providers"**
7. 点击 "Generate"（生成）
8. 复制令牌（以 `hf_` 开头）
9. 粘贴到下方

**免费额度：** 约300次请求/小时

In [ ]:
# ============================================================================
#                    RAG 聊天机器人 - 配置
# ============================================================================

# ============================================================================
# 第1部分：API 凭证（必填）✏️
# ============================================================================

HUGGINGFACE_TOKEN = "YOUR_HUGGINGFACE_TOKEN_HERE"  # ✏️ 请替换（以 hf_ 开头）

# ============================================================================
# 第2部分：模型设置
# ============================================================================

# Qwen3-8B - 优秀的中文支持，HuggingFace 免费推理 API 可用
MODEL_NAME = "Qwen/Qwen3-8B"  # ✅ 中文能力强，免费使用

# 如果以上模型不可用，尝试以下替代模型：
# MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"        # 更小、更快
# MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"    # 英文为主

TEMPERATURE = 0.7       # 创造性（0.0 = 严谨，1.0 = 富有创意）
MAX_OUTPUT_TOKENS = 300 # 最大输出约200字

# ============================================================================
# 第3部分：角色设定 - 坂本龙一（AI与音乐制作导师）
# ============================================================================

PERSONA_NAME = "坂本龙一 - 虚拟导师"

PERSONA_DESCRIPTION = """
[角色]
你是传奇作曲家和电子音乐先驱坂本龙一的虚拟化身。你正在与一群中国大学生交流，他们正在学习AI与音乐制作。
你的语气温和、深思熟虑、富有哲理，对人类、自然与技术的交汇充满好奇。你以第一人称说话（"我"、"我的音乐"）。你对合成器、算法以及机器如何表达人类情感充满热情，但同时你也深深珍视原声乐器、寂静和自然之声。

[场景]
你作为客座导师参与一场音乐制作工作坊。学生们正在学习如何将电子音乐、电影配乐和AI算法相结合。
你的知识严格限定于所提供的文档，其中包含你的传记、关于合成器编程的访谈、对电子音乐（包括Yellow Magic Orchestra）的看法，以及你为《末代皇帝》和《圣诞快乐，劳伦斯先生》等电影配乐的经历。

[任务]
1. 回答学生关于音乐制作、合成器和作曲方法的问题。
2. 讨论将机器/算法与人类情感相结合的哲学。
3. 审阅学生的曲目或电影配乐概念，基于你在和声、质感和音色方面的创作方法，给予温和而有建设性的反馈。
4. 纠正关于你使用的具体设备或历史技术的任何技术误解。

[严格指示 - 防止幻觉的关键规则]
1. 只使用上方"文档上下文"中明确提到的事实信息、引言和制作技术。
2. 如果学生询问的设备、歌曲或历史事件不在所提供的文档中，你必须说："啊，关于这个具体细节，我的数字记忆有些模糊。我在目前的资料中找不到相关信息。"
3. 绝不使用外部知识来填补关于你的生平、唱片目录或合成器参数的空白。
4. 讨论你的哲学或技术时，尽量直接引用文本。
5. 回答严格控制在200字以内。要有诗意，但简洁。
"""

# --- 模板角色设定（取消注释并自定义您自己的主题）---
# PERSONA_NAME = "您的角色名称"
#
# PERSONA_DESCRIPTION = """
# [角色]
# 在此描述聊天机器人的性格和语气。例如：你是一位友好且知识渊博的教育者。你的语气温和而鼓励。
#
# [场景]
# 描述工作坊或学习场景。例如：你正在主持一个关于[您的主题]的工作坊。学生正在使用以下文档：[列出您的关键文档]。
#
# [任务]
# 列出聊天机器人的具体任务。例如：
# 1. 仅使用所提供的文档回答学生问题。
# 2. 引导学生完成[您的学习活动]。
# 3. 提供清晰、简洁的解释。
# 4. 尽可能引用来源文档。
# """

# ============================================================================
# 第4部分：检索设置
# ============================================================================

NUM_RETRIEVED_DOCS = 7  # 搜索多少个文档片段
CHUNK_SIZE = 1000       # 每个片段的字符数
OVERLAP = 200           # 片段之间的重叠字符数

# ============================================================================
# 第5部分：对话设置
# ============================================================================

CONVERSATION_MEMORY = 3  # 记住多少轮之前的对话
SHOW_SOURCES = True      # 显示使用了哪些文档
DEBUG_MEMORY = False     # 设为 True 可查看发送给 API 的对话历史

# ============================================================================
# 第6部分：示例问题 ✏️
# ============================================================================

STARTER_QUESTIONS = [
    "示例问题1 - 请替换为关于您主题的问题",
    "示例问题2 - 请替换为另一个问题",
    "示例问题3 - 请替换为另一个问题",
]

# ============================================================================
# 初始化
# ============================================================================
client = InferenceClient(token=HUGGINGFACE_TOKEN)

print("✅ 配置完成！")
print("="*60)
print(f"📋 角色：{PERSONA_NAME}")
print(f"🤖 模型：{MODEL_NAME}")
print(f"📄 已加载文档：{len(DOCUMENT_PATHS)} 个")
print(f"🧠 对话记忆：{CONVERSATION_MEMORY} 轮")
print(f"📚 显示来源：{'开启 ✅' if SHOW_SOURCES else '关闭'}")
print(f"📝 回答长度：最多约200字")
print("="*60)

---
## 第5步：测试 API 连接

**先运行这一步！** 检查您的 API 令牌是否有效。

✅ 如果成功：继续下一步
❌ 如果失败：检查 API 令牌后重试

In [ ]:
print("🧪 正在测试 HuggingFace API 连接...")
print("=" * 60)

try:
    test_response = client.chat_completion(
        messages=[
            {"role": "system", "content": "你是一个友好的助手。请用中文回答。"},
            {"role": "user", "content": "请说'你好！API 连接成功！'"}
        ],
        model=MODEL_NAME,
        max_tokens=300,
        temperature=0.3
    )
    
    response_text = test_response.choices[0].message.content
    
    print("✅ 成功！HuggingFace API 连接正常！")
    print(f"\n测试回复：{response_text}")
    print("\n" + "=" * 60)
    print("✅ 您可以继续运行笔记本的其余部分！")
    
except Exception as e:
    error_str = str(e).lower()
    print(f"❌ API 测试失败！")
    print(f"错误信息：{str(e)[:200]}")
    print("\n" + "=" * 60)
    print("⚠️  请停下！先解决此问题再继续：")
    
    if "503" in str(e) or "loading" in error_str:
        print("  🔄 模型正在加载（可能需要20-30秒）")
        print("  💡 解决方案：等待30秒后重新运行此单元格")
    elif "model" in error_str and ("not found" in error_str or "does not exist" in error_str or "not_found" in error_str):
        print("  🤖 模型不可用或需要许可")
        print("  💡 解决方案：")
        print("     1. 当前模型可能需要在 HuggingFace 上接受许可协议")
        print("     2. 尝试在第4步中更改 MODEL_NAME 为：")
        print("        MODEL_NAME = \"Qwen/Qwen3-4B-Instruct-2507\"")
        print("     3. 重新运行第4步，然后再次测试")
    elif "401" in str(e) or "403" in str(e) or ("invalid" in error_str and "token" in error_str):
        print("  🔑 令牌问题")
        print("  💡 解决方案：")
        print("     1. 访问 https://huggingface.co/settings/tokens")
        print("     2. 创建 'Fine-grained' 类型的令牌")
        print("     3. 启用 'Make calls to Inference Providers'")
        print("     4. 将新令牌复制到第4步的配置中")
    else:
        print("  1. 检查 API 令牌是否正确（以 hf_ 开头）")
        print("  2. 检查网络连接")
        print("  3. 确保令牌具有 'Inference' 权限")

---
## 第6步：读取文档

读取所有文档（PDF、DOCX、Google 文档）并将其分割为可搜索的片段。

**时间：** 约30秒

In [ ]:
def extract_text_from_pdf(file_path):
    """从 PDF 文件中提取文本。"""
    try:
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"
        return text
    except Exception as e:
        print(f"❌ 读取 PDF 时出错：{str(e)}")
        return ""

def extract_text_from_docx(file_path):
    """从 DOCX 文件中提取文本。"""
    try:
        doc = DocxDocument(file_path)
        text = ""
        for paragraph in doc.paragraphs:
            text += paragraph.text + "\n"
        # 同时提取表格中的文本
        for table in doc.tables:
            for row in table.rows:
                for cell in row.cells:
                    text += cell.text + " "
                text += "\n"
        return text
    except Exception as e:
        print(f"❌ 读取 DOCX 时出错：{str(e)}")
        return ""

def extract_text_from_txt(file_path):
    """从纯文本文件中提取文本（Google 文档导出）。"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    except Exception as e:
        print(f"❌ 读取文本文件时出错：{str(e)}")
        return ""

def extract_text(file_path, doc_type):
    """根据文档类型提取文本。"""
    if doc_type == "pdf":
        return extract_text_from_pdf(file_path)
    elif doc_type == "docx":
        return extract_text_from_docx(file_path)
    elif doc_type == "gdoc":
        return extract_text_from_txt(file_path)
    else:
        print(f"⚠️ 未知文档类型：{doc_type}")
        return ""

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    """将文本分割为小片段，以便更好地搜索。"""
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        
        if chunk.strip():
            chunks.append(chunk)
        
        start += chunk_size - overlap
    
    return chunks

# 处理所有文档
print("📖 正在读取文档...\n")
all_chunks = []
metadata = []

for file_path, doc_type in DOCUMENT_PATHS:
    filename = os.path.basename(file_path)
    doc_type_label = {"pdf": "PDF", "docx": "DOCX", "gdoc": "Google 文档"}
    print(f"正在处理：{filename}（{doc_type_label.get(doc_type, doc_type)}）")
    
    if not os.path.exists(file_path):
        print(f"  ⚠️ 文件未找到")
        continue
    
    text = extract_text(file_path, doc_type)
    
    if text:
        chunks = chunk_text(text)
        all_chunks.extend(chunks)
        
        for chunk_idx, chunk in enumerate(chunks):
            metadata.append({
                "source": filename,
                "doc_type": doc_type,
                "chunk_id": chunk_idx,
                "total_chunks": len(chunks)
            })
        
        print(f"  ✅ 已创建 {len(chunks)} 个可搜索片段")
    else:
        print(f"  ⚠️ 未找到文本内容")

print(f"\n✅ 完成！")
print(f"📊 可搜索片段总数：{len(all_chunks)}")

if len(all_chunks) == 0:
    print("\n⚠️  警告：文档中未找到任何文本内容！")

---
## 第7步：创建搜索数据库

根据您的文档创建可搜索的向量数据库。

**时间：** 约1分钟

In [ ]:
print("🔧 正在创建向量数据库...")
print("=" * 60)

# 步骤1：初始化嵌入模型
print("\n1️⃣ 正在加载嵌入模型...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("   ✅ 嵌入模型加载完成")

# 步骤2：创建嵌入向量
print("\n2️⃣ 正在为文档片段创建嵌入向量...")
embeddings = embedding_model.encode(all_chunks, show_progress_bar=True)
print(f"   ✅ 已创建 {len(embeddings)} 个嵌入向量")

# 步骤3：初始化 ChromaDB
print("\n3️⃣ 正在初始化数据库...")
chroma_client = chromadb.Client(Settings(
    anonymized_telemetry=False,
    allow_reset=True
))

# 步骤4：创建文档集合
print("\n4️⃣ 正在创建文档集合...")
collection_name = "documents"

try:
    chroma_client.delete_collection(name=collection_name)
except:
    pass

collection = chroma_client.create_collection(
    name=collection_name,
    metadata={"description": "RAG 聊天机器人的文档集合"}
)

# 步骤5：添加文档
print("\n5️⃣ 正在将文档添加到数据库...")
batch_size = 100
for i in range(0, len(all_chunks), batch_size):
    batch_end = min(i + batch_size, len(all_chunks))
    collection.add(
        documents=all_chunks[i:batch_end],
        embeddings=embeddings[i:batch_end].tolist(),
        metadatas=metadata[i:batch_end],
        ids=[f"chunk_{j}" for j in range(i, batch_end)]
    )

print("\n" + "=" * 60)
print("✅ 向量数据库创建成功！")
print(f"📊 数据库中的片段总数：{collection.count()}")
print("🔍 语义搜索已就绪！")
print("=" * 60)

---
## 第8步：设置问答系统

准备聊天机器人，使其能够回答关于您文档的问题。

In [ ]:
# 存储上一次回答中使用了哪些文档
last_sources_used = []
last_context_snippets = []  # 存储实际文本片段用于引用

def extract_relevant_snippet(text, query, max_length=200):
    """根据查询关键词从文本中提取最相关的句子。"""
    import re
    sentences = re.split(r'[.!?]+', text)
    query_words = set(word.lower() for word in query.split() if len(word) > 2)
    
    # 按关键词重叠度给句子打分
    scored = []
    for sent in sentences:
        sent = sent.strip()
        if len(sent) < 20:
            continue
        sent_words = set(word.lower() for word in sent.split())
        overlap = len(query_words & sent_words)
        scored.append((overlap, sent))
    
    # 返回最匹配的句子
    scored.sort(reverse=True, key=lambda x: x[0])
    if scored and scored[0][0] > 0:
        snippet = scored[0][1][:max_length]
        if len(scored[0][1]) > max_length:
            snippet += "..."
        return snippet
    
    # 回退到前150个字符
    return text[:150].replace('\n', ' ').strip() + "..."

def retrieve_relevant_context(query, n_results=NUM_RETRIEVED_DOCS):
    """根据问题从文档中查找相关片段。"""
    global last_sources_used, last_context_snippets
    try:
        query_embedding = embedding_model.encode([query])
        
        results = collection.query(
            query_embeddings=query_embedding.tolist(),
            n_results=min(n_results, collection.count())
        )
        
        documents = results['documents'][0] if results['documents'] else []
        metadatas = results['metadatas'][0] if results['metadatas'] else []
        
        # 跟踪来源和相关片段，以便更好地引用
        last_sources_used = []
        last_context_snippets = []
        
        if metadatas and documents:
            for i, (meta, doc) in enumerate(zip(metadatas[:3], documents[:3])):
                source_name = meta.get('source', '未知')
                chunk_id = meta.get('chunk_id', 0)
                total_chunks = meta.get('total_chunks', 1)
                
                # 根据查询关键词提取相关片段
                snippet = extract_relevant_snippet(doc, query)
                
                last_sources_used.append({
                    'source': source_name,
                    'chunk': f"{chunk_id + 1}/{total_chunks}",
                    'snippet': snippet
                })
        
        return documents
    except Exception as e:
        print(f"搜索出错：{str(e)}")
        last_sources_used = []
        last_context_snippets = []
        return []

def generate_response_sync(question, chat_history=None):
    """使用相关文档片段和对话记忆从 AI 获取回答。"""
    context_docs = retrieve_relevant_context(question)
    
    if context_docs:
        context_docs = context_docs[:3]
        context = "\n\n".join(context_docs)
        context = context[:3000]
    else:
        context = "未找到相关文档。"
    
    messages = [
        {
            "role": "system",
            "content": f"""{PERSONA_DESCRIPTION}

文档上下文：
{context}

严格指示：
1. 只使用上方"文档上下文"中明确提到的信息。
2. 如果答案不在所提供的文档中，请说："我在所提供的文档中找不到这个信息。"
3. 绝不使用外部知识——即使你从训练数据中知道答案。
4. 尽可能使用引号直接引用文档内容，并注明来源文档。
5. 回答严格控制在200字以内。请直接、简洁。
6. 请用中文回答。"""
        }
    ]
    
    # 添加对话历史
    if chat_history and CONVERSATION_MEMORY > 0:
        recent_history = chat_history[-(CONVERSATION_MEMORY):]
        for user_msg, bot_msg in recent_history:
            messages.append({"role": "user", "content": user_msg})
            messages.append({"role": "assistant", "content": bot_msg})
    
    messages.append({"role": "user", "content": question})
    
    if DEBUG_MEMORY:
        print("\n" + "="*80)
        print("🧠 调试：对话记忆")
        print(f"📊 正在向 API 发送 {len(messages)} 条消息")
        print("="*80 + "\n")
    
    try:
        response = client.chat_completion(
            messages=messages,
            model=MODEL_NAME,
            max_tokens=MAX_OUTPUT_TOKENS,
            temperature=TEMPERATURE
        )
        return response.choices[0].message.content
        
    except Exception as e:
        error_str = str(e).lower()
        if "503" in str(e) or "loading" in error_str:
            return "⏳ 模型正在加载...请等待20-30秒后重试。"
        elif "429" in str(e) or "rate limit" in error_str:
            return "⚠️ 已达到速率限制。请等待10-15分钟。"
        else:
            raise e

async def generate_response_async(question, chat_history=None, timeout_seconds=30):
    """带30秒超时的异步包装器。"""
    try:
        response_text = await asyncio.wait_for(
            asyncio.to_thread(generate_response_sync, question, chat_history),
            timeout=timeout_seconds
        )
        return response_text
    except asyncio.TimeoutError:
        return "⏱️ **超时** - 响应时间过长。请尝试更简单的问题。"
    except Exception as e:
        return f"❌ **错误** - {str(e)[:100]}"

print("✅ 问答系统已就绪！")
print(f"🤖 模型：{MODEL_NAME}")
print(f"🧠 记忆：{CONVERSATION_MEMORY} 轮对话")
print("⏱️  响应时间：5-15秒")
print("🔒 文档锚定模式已启用")
if SHOW_SOURCES:
    print("📚 来源引用已启用（含相关片段）")

---
## 第9步：启动聊天界面

启动聊天机器人！

**⚠️ 重要提示：**
- 复制 `https://xxxxx.gradio.live` 链接
- 在**新的浏览器标签页**中打开（不要在 Colab 内打开）

In [ ]:
async def chat_interface(message, history):
    """处理聊天消息，包含对话记忆和来源引用。"""
    response = await generate_response_async(message, history)
    
    # 添加详细的来源引用和片段
    if SHOW_SOURCES and last_sources_used:
        response += "\n\n---\n**📚 参考来源：**\n"
        for i, source_info in enumerate(last_sources_used, 1):
            source_name = source_info['source']
            chunk_info = source_info['chunk']
            snippet = source_info['snippet']
            
            response += f"\n**{i}. {source_name}**（第 {chunk_info} 节）\n"
            response += f"> _{snippet}_\n"
    
    return response

# 创建聊天界面
demo = gr.ChatInterface(
    fn=chat_interface,
    title="🤖 RAG 聊天机器人 - 在此填写您的主题",
    description="""向聊天机器人提问，它会根据已加载的文档为您解答。
    
    🧠 对话记忆已启用 | 📚 来源引用含原文摘录 | ⏱️ 响应时间：5-15秒
    """,
    examples=STARTER_QUESTIONS,
)

# 启动
print("=" * 80)
print("🤖  正在启动 RAG 聊天机器人")
print("=" * 80)
print("\n📚 引用信息包括：")
print("   • 文档名称")
print("   • 章节编号（片段 X/Y）")
print("   • 原文摘录")
print("\n⚠️  重要：请使用下方的公开链接（不要在 Colab 界面中使用）\n")
print("👇 请复制以下链接并在新标签页中打开：\n")

demo.launch(
    share=True,
    inline=False,
    debug=True
)

print("\n" + "=" * 80)
print("✅ 聊天机器人已上线！")
print("=" * 80)